### Experimental Code for DataCollector() main Class

First Language Source is the Huggingface wikipedia dataset.

In [12]:
import random

from datasets import load_dataset
import requests

LANGUAGES = {
    "english": "en",
    "hindi": "hi",
    "spanish": "es",
    "french": "fr",
    "german": "de",
    "japanese": "ja",
}

COUNTRY_CODE = LANGUAGES["hindi"]
SHARD_CODE = f"20231101.{COUNTRY_CODE}"

response = requests.get(
    "https://datasets-server.huggingface.co/parquet",
    params={"dataset": "wikimedia/wikipedia"}
)

data = response.json()
FILES = [f for f in data["parquet_files"] if f["config"] == SHARD_CODE]

print(f"{COUNTRY_CODE} has {len(FILES)} shards.")

for f in FILES:
    print(f["filename"], f["url"])

hi has 2 shards.
0000.parquet https://huggingface.co/datasets/wikimedia/wikipedia/resolve/refs%2Fconvert%2Fparquet/20231101.hi/train/0000.parquet
0001.parquet https://huggingface.co/datasets/wikimedia/wikipedia/resolve/refs%2Fconvert%2Fparquet/20231101.hi/train/0001.parquet


Using a single random seed to make the textual data randomized, but also reproducible.

In [13]:
RANDOM_SEED = 42
shard_random_seed = random.Random(
    RANDOM_SEED
)

shard_indices = list(range(len(FILES)))
shard_random_seed.shuffle(shard_indices)

print(f"Shard Order: {shard_indices}")

Shard Order: [1, 0]


In [16]:
target = 1 # in MB

target_bytes = target * 1024 * 1024

all_shards = [f"https://huggingface.co/datasets/wikimedia/wikipedia/resolve/refs%2Fconvert%2Fparquet/{SHARD_CODE}/train/{idx:04d}.parquet" for idx in shard_indices]
all_shards

['https://huggingface.co/datasets/wikimedia/wikipedia/resolve/refs%2Fconvert%2Fparquet/20231101.hi/train/0001.parquet',
 'https://huggingface.co/datasets/wikimedia/wikipedia/resolve/refs%2Fconvert%2Fparquet/20231101.hi/train/0000.parquet']

In [17]:
row_seed = RANDOM_SEED + 1
collected_text = []
collected_bytes = 0
collected_article_titles = []

for shard in all_shards:
    
    if collected_bytes >= target_bytes:
        break
    
    print(f"Using shard {shard}:")
    
    dataset = load_dataset(
        "parquet",
        data_files={
            "train": shard
        },
        split="train",
        streaming=True
    )
    
    dataset = dataset.shuffle(
        seed=row_seed,
        buffer_size=10_000
    )
    
    shard_bytes = 0
    shard_articles = 0
    
    for row in dataset:
        text = row["text"]
        title = row["title"]
        
        if not text:
            continue
        
        text_bytes = len(text.encode("utf-8"))
        
        collected_text.append(text)
        collected_bytes += text_bytes
        shard_articles += 1
        shard_bytes += text_bytes
        
        collected_article_titles.append(title)
        
        if collected_bytes >= target_bytes:
            break
        
    print("Collected from shard: "
          f"{shard_bytes / (1024 * 1024):.2f} MB")
    
    print("Total collected: "
          f"{collected_bytes / (1024 * 1024):.2f} MB")
    

text = "\n\n".join(collected_text)

print(collected_article_titles)
text

Using shard https://huggingface.co/datasets/wikimedia/wikipedia/resolve/refs%2Fconvert%2Fparquet/20231101.hi/train/0001.parquet:
Collected from shard: 1.15 MB
Total collected: 1.15 MB
['आंशिक अवकल समीकरण', 'कचोलिया', 'टाइम (अंग्रेज़ी पत्रिका)', 'अरब देश', 'दया कि\u200dशोर हाजरा', 'जिब्राल्टर 2', 'राजस्थान की रूपरेखा', 'सत्यपाल सिंह', 'उस्ताख ल सर', 'विकासनगर', 'तू मेरा हीरो', 'जानूस (चंद्रमा)', 'विश्लेषणात्मक विधिशास्त्र', 'विद्याधर (बहुविकल्पी)', 'सूरजपाल चौहान', 'रव', 'उत्तरी लखीमपुर', 'दादरी विधानसभा निर्वाचन क्षेत्र, हरियाणा', 'जयपुर-वैभवम', 'सामाजिक न्याय', 'क्लब', 'अस्कोट कस्तूरी मृग अभयारण्य', 'दि एसेंस ऑफ बुद्धिज़्म', 'तेवेरगा का सान पेदरो गिरजाघर', 'आकाश खुराना', 'सीमाओं की सूची', 'रामदास तडस', 'संख्यात्मक समाकलन', 'सांता मारिया गिरजाघर (सेबरायो)', 'फाइव प्वाइंट समवन', 'प्रदीप (पत्र)', 'ऐडमिरल', 'जॉन कॉर्नफोर्थ', 'नैरोबी के शॉपिंग मॉल में गोलाबारी', 'द क्रॉनिकल्स ऑफ़ नार्निया: प्रिंस कैस्पियन', 'क्रिस यंग (बेसबॉल खिलाड़ी)', 'ट्रफ़ैलगर कब्रिस्तान', 'कलराज मिश्र', 'पम्पोश भट', '

'गणित में आंशिक अवकल समीकरण वो अवकल समीकरणें होती हैं जिनमें बहुचर फलन और उनके आंशिक अवकल होते हैं। (यह साधारण अवकल समीकरणों से भिन्न है जिनमें एक ही चर और उसके अवकलों में बंटा हुआ होता है। आंशिक अवकल समीकरणों का उपयोग उन समस्याओं को हल करने में प्रयुक्त किया जाता है जो विभिन्न स्वतंत्र चरों की फलन होती हैं एवं जिन्हें साधारणतया हल कर सकते हैं अथवा हल करने के लिए अभिकलित्र प्रोग्राम बनाया जा सके।\n\nआंशिक अवकल समीकरणो का उपयोग विभिन्न दृष्टिगत घटनाओं यथा ध्वनि, ऊष्मा, स्थिरवैद्युतिकी, विद्युत-गतिकी, द्रव का प्रवाह, प्रत्यास्थता या प्रमात्रा यान्त्रिकी को समझने में किया जा सकता है। ये पृथक प्रतीत होने वाली प्रक्रियाओं को आंशिक अवकल समीकरणों के रूप में सूत्रित किया जा सकता है।\n\nउदाहरण \nनिम्नलिखित समीकरण, आंशिक अवकल समीकरण का एक उदाहरण है-\n\n,\n\nजिसका सामान्य हल (जनरल सलूसन) निम्नलिखित है-\n\n.\n\nजहाँ  और  यादृच्छिक (आर्बिट्रेरी) फलन हैं।\n\nलाप्लास का समीकरण\n\nतनी हुई डोरी का कम्पन\n\nऐडवेक्सन (advection) समीकरण\n\nलैंगमूर (Langmuir ) का समीकरण\n\nस्टोक्स (Stokes) का समीकरण \n\n,\

So huggingface wikipedia pipeline is almost complete.

Two caveats:
 1. More Languages can be added.
 2. Suppose the total byte size of a language < target, in that case, we can add a check and make target == available_size.


Now, for Programming Languages:

In [23]:
import random 
import requests

PROGRAMMING_REPOS = {
    "python": [
        "python/cpython",
        "numpy/numpy",
        "pallets/flask",
        "psf/requests",
        "django/django",
    ],

    "cpp": [
        "tensorflow/tensorflow",
        "opencv/opencv",
        "llvm/llvm-project",
        "bitcoin/bitcoin",
        "catchorg/Catch2",
    ],

    "javascript": [
        "facebook/react",
        "nodejs/node",
        "expressjs/express",
        "axios/axios",
        "vuejs/core",
    ],

    "java": [
        "spring-projects/spring-framework",
        "apache/kafka",
        "elastic/elasticsearch",
        "google/guava",
        "apache/maven",
    ],

    "rust": [
        "rust-lang/rust",
        "tokio-rs/tokio",
        "serde-rs/serde",
        "clap-rs/clap",
        "BurntSushi/ripgrep",
    ],
}

PROGRAMMING_EXTENSIONS = {
    "python": {"py"},
    "cpp": {"cpp", "cc", "cxx", "h", "hpp"},
    "javascript": {"js", "jsx", "mjs", "cjs"},
    "java": {"java"},
    "rust": {"rs"},
}

target_language = "python"
target_size = 10
RANDOM_SEED = 1

target_bytes = target_size * 1024 * 1024

headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2026-03-10",
}

respositories = PROGRAMMING_REPOS[target_language].copy()
repo_seed = random.Random(RANDOM_SEED)
repo_seed.shuffle(respositories)

file_seed = random.Random(RANDOM_SEED + 1)
extensions = PROGRAMMING_EXTENSIONS[target_language]
collected_files = []
collected_bytes = 0

# PROCESSING AN INDIVIDUAL REPO
sample_repo = respositories[0]

owner, repo = sample_repo.split("/")
print(f"Sample Repo: {sample_repo}")

repo_url = (
    f"https://api.github.com/repos/"
    f"{owner}/{repo}"
)

response = requests.get(
    repo_url,
    headers=headers
)

response.raise_for_status()
repo_data = response.json()
default_branch = repo_data[
    "default_branch"
]

tree_url = (
    f"https://api.github.com/repos/"
    f"{owner}/{repo}/git/trees/"
    f"{default_branch}"
)

response = requests.get(
    tree_url,
    params={"recursive": "1"},
    headers=headers
)

response.raise_for_status()

tree_data = response.json()

if tree_data.get("truncated", False):
    print("Warning: repository tree truncated.")
    
tree_data

Sample Repo: pallets/flask


{'sha': 'd318b683471101618febed18996405ad26462110',
 'url': 'https://api.github.com/repos/pallets/flask/git/trees/d318b683471101618febed18996405ad26462110',
 'tree': [{'path': '.devcontainer',
   'mode': '040000',
   'type': 'tree',
   'sha': '119c3135411fa2ed205593d56dde8e1281d22f89',
   'url': 'https://api.github.com/repos/pallets/flask/git/trees/119c3135411fa2ed205593d56dde8e1281d22f89'},
  {'path': '.devcontainer/devcontainer.json',
   'mode': '100644',
   'type': 'blob',
   'sha': '45198266c6a481104f76d07778086e1df9164ead',
   'size': 434,
   'url': 'https://api.github.com/repos/pallets/flask/git/blobs/45198266c6a481104f76d07778086e1df9164ead'},
  {'path': '.devcontainer/on-create-command.sh',
   'mode': '100755',
   'type': 'blob',
   'sha': 'eaebea61856f7f6bde03a0deb1c7098fb085ca29',
   'size': 165,
   'url': 'https://api.github.com/repos/pallets/flask/git/blobs/eaebea61856f7f6bde03a0deb1c7098fb085ca29'},
  {'path': '.editorconfig',
   'mode': '100644',
   'type': 'blob',
   'sh

In [ ]:
files = []
actual_tree_structure = tree_data['tree']

for item in actual_tree_structure:
    
    if item['type'] != 'blob':
        continue
    
    path = item['path']
    if "." in path:
        extension = path.rsplit(".", 1)[1]
        
    if extension.lower() not in extensions:
        continue
    
    files.append(path)
    
print("All Python files: ")
print(files)

All Python files: 
['docs/conf.py', 'examples/celery/make_celery.py', 'examples/celery/src/task_app/__init__.py', 'examples/celery/src/task_app/tasks.py', 'examples/celery/src/task_app/views.py', 'examples/javascript/js_example/__init__.py', 'examples/javascript/js_example/views.py', 'examples/javascript/tests/conftest.py', 'examples/javascript/tests/test_js_example.py', 'examples/tutorial/flaskr/__init__.py', 'examples/tutorial/flaskr/auth.py', 'examples/tutorial/flaskr/blog.py', 'examples/tutorial/flaskr/db.py', 'examples/tutorial/tests/conftest.py', 'examples/tutorial/tests/test_auth.py', 'examples/tutorial/tests/test_blog.py', 'examples/tutorial/tests/test_db.py', 'examples/tutorial/tests/test_factory.py', 'src/flask/__init__.py', 'src/flask/__main__.py', 'src/flask/app.py', 'src/flask/blueprints.py', 'src/flask/cli.py', 'src/flask/config.py', 'src/flask/ctx.py', 'src/flask/debughelpers.py', 'src/flask/globals.py', 'src/flask/helpers.py', 'src/flask/json/__init__.py', 'src/flask/js

In [25]:
file_seed.shuffle(files)

for path in files:
    
    if collected_bytes >= target_bytes:
        break
    
    raw_url = (
        f"https://raw.githubusercontent.com/"
        f"{owner}/{repo}/"
        f"{default_branch}/{path}"
    )
    
    response = requests.get(
        raw_url,
        headers=headers
    )
    
    if response.status_code != 200:
        print(f"Skipping {path} because of Status Code {response.status_code}")
        continue
    
    content = response.content
    
    try:
        text = content.decode("utf-8")
    except UnicodeDecodeError:
        print(f"Skipping non-utf-8 file: {path}")
        continue
    
    if not text.strip():
        continue
    
    text_bytes = len(text.encode("utf-8"))
    
    remaining = target_bytes - collected_bytes
    
    if text_bytes > remaining:
        
        encoded = text.encode("utf-8")
        text = encoded[:remaining].decode(
            "utf-8",
            errors="ignore"
        )
        
        text_bytes = len(text.encode("utf-8"))
        
    collected_files.append(text)
    collected_bytes += text_bytes
    
    print(
        f"  {path}: "
        f"{text_bytes / (1024 * 1024):.2f} MB "
        f"| total: "
        f"{collected_bytes / (1024 * 1024):.2f} MB"
    )
    
corpus = "\n\n".join(collected_files)
corpus

  docs/conf.py: 0.00 MB | total: 0.00 MB
  src/flask/logging.py: 0.00 MB | total: 0.01 MB
  tests/test_regression.py: 0.00 MB | total: 0.01 MB
  examples/javascript/js_example/views.py: 0.00 MB | total: 0.01 MB
  tests/test_apps/cliapp/app.py: 0.00 MB | total: 0.01 MB
  src/flask/wrappers.py: 0.01 MB | total: 0.02 MB
  examples/celery/src/task_app/__init__.py: 0.00 MB | total: 0.02 MB
  tests/test_json.py: 0.01 MB | total: 0.02 MB
  examples/tutorial/tests/test_factory.py: 0.00 MB | total: 0.03 MB
  examples/tutorial/tests/conftest.py: 0.00 MB | total: 0.03 MB
  tests/test_cli.py: 0.02 MB | total: 0.05 MB
  examples/tutorial/flaskr/__init__.py: 0.00 MB | total: 0.05 MB
  examples/javascript/js_example/__init__.py: 0.00 MB | total: 0.05 MB
  tests/test_apps/helloworld/wsgi.py: 0.00 MB | total: 0.05 MB
  src/flask/__main__.py: 0.00 MB | total: 0.05 MB
  src/flask/debughelpers.py: 0.01 MB | total: 0.05 MB
  tests/test_testing.py: 0.01 MB | total: 0.06 MB
  src/flask/__init__.py: 0.00 MB |

'import packaging.version\nfrom pallets_sphinx_themes import get_version\nfrom pallets_sphinx_themes import ProjectLink\n\n# Project --------------------------------------------------------------\n\nproject = "Flask"\ncopyright = "2010 Pallets"\nauthor = "Pallets"\nrelease, version = get_version("Flask")\n\n# General --------------------------------------------------------------\n\ndefault_role = "code"\nextensions = [\n    "sphinx.ext.autodoc",\n    "sphinx.ext.extlinks",\n    "sphinx.ext.intersphinx",\n    "sphinxcontrib.log_cabinet",\n    "sphinx_tabs.tabs",\n    "pallets_sphinx_themes",\n]\nautodoc_member_order = "bysource"\nautodoc_typehints = "description"\nautodoc_preserve_defaults = True\nextlinks = {\n    "issue": ("https://github.com/pallets/flask/issues/%s", "#%s"),\n    "pr": ("https://github.com/pallets/flask/pull/%s", "#%s"),\n    "ghsa": ("https://github.com/pallets/flask/security/advisories/GHSA-%s", "GHSA-%s"),\n}\nintersphinx_mapping = {\n    "python": ("https://docs.

In [26]:
print(corpus)

import packaging.version
from pallets_sphinx_themes import get_version
from pallets_sphinx_themes import ProjectLink

# Project --------------------------------------------------------------

project = "Flask"
copyright = "2010 Pallets"
author = "Pallets"
release, version = get_version("Flask")

# General --------------------------------------------------------------

default_role = "code"
extensions = [
    "sphinx.ext.autodoc",
    "sphinx.ext.extlinks",
    "sphinx.ext.intersphinx",
    "sphinxcontrib.log_cabinet",
    "sphinx_tabs.tabs",
    "pallets_sphinx_themes",
]
autodoc_member_order = "bysource"
autodoc_typehints = "description"
autodoc_preserve_defaults = True
extlinks = {
    "issue": ("https://github.com/pallets/flask/issues/%s", "#%s"),
    "pr": ("https://github.com/pallets/flask/pull/%s", "#%s"),
    "ghsa": ("https://github.com/pallets/flask/security/advisories/GHSA-%s", "GHSA-%s"),
}
intersphinx_mapping = {
    "python": ("https://docs.python.org/3/", None),
    "werk

Here, we basically downloaded all python code from the flask repository (only ~0.6 MB).

In [10]:
from gdeltdoc import GdeltDoc, Filters
import requests

GDELT_LANGUAGE_KEYMAP_SOURCE = "https://data.gdeltproject.org/api/v2/guides/LOOKUP-LANGUAGES.TXT"

response = requests.get(GDELT_LANGUAGE_KEYMAP_SOURCE)
raw_text = response.content.decode("utf-8")
keymap = raw_text.split("\n")

language_code = {}

for row in keymap:
    elements = row.split("\t")
    
    if len(elements) > 1:
    
        key = elements[1]
        value = elements[0]
        language_code[key.lower()] = value

language_code["english"] = "eng"

target_language = "english"

gd = GdeltDoc()

if target_language == "english":
    filter = Filters(
        timespan="2h",
        language="eng"
    )
    
else:
    filter = Filters(
        timespan="2h",
        language=language_code[target_language]
    )
    
articles = gd.article_search(filter)

print(type(articles))

ReadTimeout: HTTPSConnectionPool(host='api.gdeltproject.org', port=443): Read timed out. (read timeout=None)

In [5]:
print(articles.shape)

articles.head()

(0, 0)


""


In [1]:
RSS_FEEDS = {
    "english": [
        "https://feeds.bbci.co.uk/news/rss.xml",
    ],

    "hindi": [
        "https://feeds.bbci.co.uk/hindi/rss.xml",
    ],

    "japanese": [
        "https://feeds.bbci.co.uk/japanese/rss.xml",
    ],

    "french": [
        "https://www.france24.com/en/rss"
    ],

    "german": [
        "https://www.deutschland.de/en/feed-news/rss.xml"
    ],

    "spanish": [
        "https://feeds.bbci.co.uk/mundo/rss.xml"
    ],
}

target_language = "spanish"

import feedparser
from newspaper import Article
import nltk

# nltk.download("punkt")
# nltk.download("punkt_tab")

rss_url = RSS_FEEDS[target_language][0]
feed = feedparser.parse(rss_url)

articles = []

for entry in feed.entries:
    article = Article(entry.link)
    
    try:
        article.download()
        article.parse()
        
        article.nlp()
        
        parsed = {
            "title": article.title,
            "text": article.text,
            "keywords": article.keywords
        }
        
        articles.append(parsed)
        
    except Exception as e:
        print(f"Error processing article {entry.link}: {str(e)}")
        
articles
        

Error processing article https://www.bbc.com/mundo/articles/c4gqmz701vmo?at_medium=RSS&at_campaign=rss: Article `download()` failed with HTTPSConnectionPool(host='www.bbc.com', port=443): Max retries exceeded with url: /mundo/articles/c4gqmz701vmo?at_medium=RSS&at_campaign=rss (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001B51C06BE10>, 'Connection to www.bbc.com timed out. (connect timeout=7)')) on URL https://www.bbc.com/mundo/articles/c4gqmz701vmo?at_medium=RSS&at_campaign=rss


[{'title': '"Colombia tiene una paranoia tras perder Panamá en 1903 que la ha llevado al aislacionismo en el Caribe, donde podría ser un motor"',
  'text': '"Colombia tiene una paranoia tras perder Panamá en 1903 que la ha llevado al aislacionismo en el Caribe, donde podría ser un motor"\n\nFuente de la imagen, Richi dos punto cero / Filbo Pie de foto, Cristina Bendek es una escritora colombiana de la isla de San Andrés.\n\nAutor, Título del autor, BBC News Mundo @HayFestivalQuerétaro\n\nFecha de publicación 21 minutos\n\nTiempo de lectura: 7 min\n\nSan Andrés, Providencia y Santa Catalina son tres islas colombianas en el Caribe con arena prístina, mar de cristal y una población que habla un creole influenciado por inglés, español y lenguas africanas.\n\nEstán más cerca de Nicaragua que de Colombia: a 110 kilómetros del primero y 720 del segundo. Eso ha originado disputas diplomáticas y de soberanía entre ambas naciones por más de un siglo.\n\nDos eventos recientes afectaron a sus 80.0

In [4]:
combined_text = ""

for article in articles:
    combined_text += article.get("title", "")
    combined_text += article.get("text", "")
    
    combined_text += " ".join(article.get("keywords", ""))
    
total_bytes = len(combined_text.encode("utf-8"))
total_bytes

total_size_in_mb = (total_bytes / (1024 * 1024))
total_size_in_mb

0.33376216888427734

So the DataCollector Class has been made, now let us test it out.

In [5]:
import os

root = os.path.join(os.getcwd(), "..")
scripts = os.path.join(root, "scripts")

import os
import sys

root = os.path.abspath(os.path.join(os.getcwd(), ".."))
scripts = os.path.abspath(os.path.join(root, "scripts"))

if scripts not in sys.path:
    sys.path.insert(0, scripts)

from data_collection import DataCollector

output_dir = os.path.join(root, "output_files")
trial_dir = os.path.join(output_dir, "trial_run_data_collection")

os.makedirs(trial_dir, exist_ok=True)

output_path = os.path.join(trial_dir, "sample_output.txt")
report_path = os.path.join(trial_dir, "sample_report.txt")

dataObject = DataCollector(output_path, report_path)
# dataObject.add_language("english", 5, 12)
# dataObject.add_language("german", 5, 12)
# dataObject.add_language("japanese", 4, 12)

dataObject.add_programming_language("javascript", 2, 10)
dataObject.add_programming_language("java", 2, 10)
dataObject.add_programming_language("rust", 2, 10)
dataObject.add_programming_language("cpp", 2, 10)

Iterating over Repositories::  20%|██        | 1/5 [01:27<05:51, 87.78s/it]
